# Pasado `gpu-hybrid`: Colab merenje CPU fallbacka i `sync_free` hibrida

Pre pokretanja izaberi **Runtime > Change runtime type > GPU** (poželjno A100).

Notebook klonira granu `gpu-hybrid` (`a4dd0450`), pokreće:
1. `experiments/measure_hybrid_fallback.py` (`big`, `3layer`, `4layer`, `5layer` na CUDA `float64` i `big` na CPU `float64`),
2. `experiments/measure_sync_free_hybrid.py` (`baseline` naspram `sync_free` verzije bez nepotrebnih GPU/CPU sinhronizacija na `big` u `float64` i `float32`),
3. prikazuje tabele i pakuje sve CSV i log fajlove u `/content/gpu_hybrid_fallback_results.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)

In [ ]:
import os
import shutil
import subprocess

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
EXPECTED = "a4dd04509b4b8c84b526fce11239446ed70a368c"

shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run([
    "git", "-c", "http.version=HTTP/1.1", "clone",
    "--branch", "gpu-hybrid", "--single-branch", URL, REPO,
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(clone.stdout)

if clone.returncode != 0:
    print("Standardni clone nije uspeo; pokušavam fetch fallback.")
    shutil.rmtree(REPO, ignore_errors=True)
    os.makedirs(REPO)
    subprocess.run(["git", "-C", REPO, "init"], check=True)
    subprocess.run([
        "git", "-C", REPO, "-c", "http.version=HTTP/1.1",
        "fetch", "--depth", "1", URL, "refs/heads/gpu-hybrid",
    ], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"], check=True)

head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", head)
assert head == EXPECTED, f"Pogrešan commit: {head} (očekivano {EXPECTED})"
assert not subprocess.check_output(["git", "-C", REPO, "status", "--short"], text=True).strip()
os.chdir(REPO)

In [ ]:
%pip install -q scikit-learn

## 1. Merenje CPU fallbacka u produkcionom hibridu (`measure_hybrid_fallback.py`)

Meri `big` na CUDA (`float64`), `big` na CPU (`float64`) i `3layer`/`4layer`/`5layer` na CUDA (`float64`) sa 3 slike, epsilon indeksima `0, 8, 15`, `seed=7`, 5 warm-up prolaza i 10 merenih ponavljanja.

In [ ]:
runs_fallback = [
    ("big", "cuda", "float64", "/content/hybrid_fallback_big_cuda_fp64"),
    ("big", "cpu", "float64", "/content/hybrid_fallback_big_cpu_fp64"),
    ("3layer", "cuda", "float64", "/content/hybrid_fallback_3layer_cuda_fp64"),
    ("4layer", "cuda", "float64", "/content/hybrid_fallback_4layer_cuda_fp64"),
    ("5layer", "cuda", "float64", "/content/hybrid_fallback_5layer_cuda_fp64"),
]

for net_name, dev_name, dt_name, prefix in runs_fallback:
    cmd = f"""
    set -o pipefail
    PYTHONUNBUFFERED=1 python experiments/measure_hybrid_fallback.py \
      --network {net_name} \
      --device {dev_name} \
      --dtype {dt_name} \
      --num-images 3 \
      --eps-indices 0 8 15 \
      --seed 7 \
      --warmup 5 \
      --repeats 10 \
      --out {prefix}.csv \
      2>&1 | tee {prefix}.log
    """
    res = subprocess.run(["bash", "-lc", cmd], cwd=REPO)
    assert res.returncode == 0, f"Pad pri pokretanju {prefix}"

## 2. Poređenje `baseline` vs `sync_free` hibrida (`measure_sync_free_hybrid.py`)

Naizmenično meri produkcioni hibrid (`baseline`) i verziju sa uklonjene 4 nepotrebne sinhronizacije u sigmoid sloju (`sync_free`), uz proveru bit-identičnosti izlaznih tenzora.

In [ ]:
runs_sync_free = [
    ("big", "cuda", "float64", "/content/sync_free_big_cuda_fp64"),
    ("big", "cuda", "float32", "/content/sync_free_big_cuda_fp32"),
]

for net_name, dev_name, dt_name, prefix in runs_sync_free:
    cmd = f"""
    set -o pipefail
    PYTHONUNBUFFERED=1 python experiments/measure_sync_free_hybrid.py \
      --network {net_name} \
      --device {dev_name} \
      --dtype {dt_name} \
      --num-images 3 \
      --eps-indices 0 8 15 \
      --seed 7 \
      --warmup 5 \
      --repeats 10 \
      --out {prefix}.csv \
      2>&1 | tee {prefix}.log
    """
    res = subprocess.run(["bash", "-lc", cmd], cwd=REPO)
    assert res.returncode == 0, f"Pad pri pokretanju {prefix}"

## 3. Pregled tabela u notebooku

In [ ]:
import pandas as pd

df_sf64 = pd.read_csv("/content/sync_free_big_cuda_fp64.csv")
pass_group = (
    df_sf64.groupby(["variant", "image_id", "epsilon_index", "repeat"], as_index=False)
    .agg({
        "precise_total_ms": "first",
        "layer_total_ms": "sum",
        "fallback_total_ms": "sum",
        "d2h_x_ms": "sum",
        "d2h_z_ms": "sum",
        "design_ms": "sum",
        "gelsd_ms": "sum",
        "h2d_ms": "sum",
    })
)
pass_group["affine_and_other_ms"] = pass_group["precise_total_ms"] - pass_group["layer_total_ms"]
pass_group["fallback_share_pct"] = pass_group["fallback_total_ms"] / pass_group["precise_total_ms"] * 100.0

print("=== Medijana po prolazu na CUDA float64 (big): baseline vs sync_free ===")
display(
    pass_group.groupby("variant")[
        [
            "precise_total_ms",
            "affine_and_other_ms",
            "layer_total_ms",
            "fallback_total_ms",
            "d2h_x_ms",
            "d2h_z_ms",
            "design_ms",
            "gelsd_ms",
            "h2d_ms",
            "fallback_share_pct",
        ]
    ].median().round(3)
)

print("\n=== Medijana po sloju na CUDA float64 (big) ===")
display(
    df_sf64.groupby(["variant", "layer"])[
        [
            "d2h_x_ms",
            "d2h_z_ms",
            "design_ms",
            "gelsd_ms",
            "h2d_ms",
            "fallback_total_ms",
            "layer_total_ms",
            "fallback_share_layer",
            "fallback_share_precise",
        ]
    ].median().round(3)
)

## 4. Preuzimanje svih CSV i log fajlova

In [ ]:
import glob
from zipfile import ZipFile
from google.colab import files

paths = sorted(
    glob.glob("/content/hybrid_fallback_*.csv")
    + glob.glob("/content/hybrid_fallback_*.log")
    + glob.glob("/content/sync_free_*.csv")
    + glob.glob("/content/sync_free_*.log")
)
archive = "/content/gpu_hybrid_fallback_results.zip"

with ZipFile(archive, "w") as output:
    for path in paths:
        if os.path.exists(path):
            output.write(path, arcname=os.path.basename(path))

print("Arhivirani fajlovi:", [os.path.basename(p) for p in paths])
files.download(archive)